# 간/종양 CT 세그멘테이션 — LiTS 2017 불균형 손실 함수 비교

## [사전 조사] SoTA 참고 (rules.md §5-0)

| 항목 | 내용 |
|------|------|
| 데이터셋 | LiTS 2017 Challenge (131 CT 볼륨, 공개 라벨) |
| 모달리티 | CT (Axial slice, Grayscale) |
| 태스크 | 3-class: BG(0) / Liver(1) / Tumor(2) |
| 불균형 | BG >> Liver >> **Tumor** (Tumor ≈ 전체의 0.5~2%) |
| SoTA (3D nnU-Net) | Liver Dice ≈ 0.963, Tumor Dice ≈ 0.702~0.739 |
| **선택 모델 (2D)** | **U-Net++ (ResNet50, ImageNet pretrained)** |
| 2D 참고 성능 | Liver Dice ≈ 0.95, Tumor Dice ≈ 0.65~0.74 |
| BG 포함 학습 | **O** (CT는 MRI와 달리 BG 포함 학습이 관행) |
| HU Windowing | [-100, 250] (간·종양 구조 포착 최적 범위) |

**연구 목적**: U-Net++ 모델 구조 고정, **손실 함수만 교체**하여 LWCE 계열 효과 측정  
**비교 Loss**: `ce_dice`, `wce_dice`, `lwce_dice`, `plwce_dice`, `cb_dice`  
**평가 지표**: Liver Dice, Tumor Dice, mDice (BG 제외)

---

## [project-planner] 실험 계획

| 단계 | 내용 | 완료 기준 |
|------|------|----------|
| 0 | 환경 설정 | device 확인 |
| 1 | 데이터 다운로드 + 슬라이스 전처리 + DataLoader | `len(slice_files) > 1000` |
| 2 | 클래스 비율 계산 | `class_counts = [bg, liver, tumor]` 출력 |
| 3 | U-Net++ 모델 + 유틸리티 함수 | `build_model()` 호출 성공 |
| 4 | Optuna alpha 탐색 | `best_alpha_plwce`, `best_alpha_pwce` 확보 |
| 5 | 5종 Loss 비교 학습 | `all_results` 딕셔너리 완성 |
| 6 | 시각화 + 평가 + JSON/Excel 저장 | 파일 저장 확인 |

### 리스크 및 대응
| 리스크 | 대응 |
|--------|------|
| Kaggle 다운로드 실패 | 수동으로 `/tmp/lits_raw/` 에 .nii 파일 배치 |
| 슬라이스 전처리 재실행 | `/tmp/lits_slices/` 존재 시 자동 스킵 |
| GPU 메모리 부족 | `BATCH_SIZE` 줄이기 (기본 16 → 8) |

In [1]:
# ── Cell 0: 환경 설정 + 패키지 설치 ──────────────────────────────────────────
import subprocess, sys

for pkg in ['segmentation-models-pytorch', 'optuna', 'nibabel', 'openpyxl', 'kagglehub']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import os, warnings, json, random, glob
warnings.filterwarnings('ignore')

import numpy as np
import nibabel as nib
import cv2
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import kagglehub

# custom_losses 경로 (rules.md §6-1)
sys.path.insert(0, '/root/imbalanced-data-LWCE/medical_data')
from custom_losses import get_loss_function

# ── 실험 설정 ─────────────────────────────────────────────────────────────────
DOMAIN      = 'lits'
NUM_CLASSES = 3
CLASS_NAMES = ['Background', 'Liver', 'Tumor']
IMG_SIZE    = 256
BATCH_SIZE  = 16     # GPU 메모리 부족 시 8로 줄이기
NUM_WORKERS = 4
SEED        = 42
HU_MIN, HU_MAX = -100, 250   # liver CT windowing

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── 결과 저장 경로 ───────────────────────────────────────────────────────────
RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/results/LiTS_Liver_Tumor'
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('환경 설정 완료')

Device: cuda
환경 설정 완료


In [2]:
# ── Cell 1: 데이터 다운로드 + 슬라이스 전처리 + Dataset + DataLoader ──────────

# 1-1. 다운로드 (kagglehub)
print('LiTS 데이터셋 다운로드 중...')
raw_path = kagglehub.dataset_download('andrewmvd/liver-tumor-segmentation')
print(f'Dataset path: {raw_path}')

# 1-2. 볼륨/라벨 파일 탐색 (재귀 glob, nii/nii.gz 모두)
vol_files  = sorted(glob.glob(os.path.join(raw_path, '**', 'volume-*.nii*'),  recursive=True))
mask_files = sorted(glob.glob(os.path.join(raw_path, '**', 'labels-*.nii*'), recursive=True))

# 혹시 segmentation-*.nii 형태인 경우 fallback
if not mask_files:
    mask_files = sorted(glob.glob(os.path.join(raw_path, '**', 'segmentation-*.nii*'), recursive=True))

def get_vol_idx(fp):
    """파일명에서 볼륨 인덱스 추출: volume-27.nii → 27"""
    return int(os.path.basename(fp).split('-')[1].split('.')[0])

vol_dict  = {get_vol_idx(fp): fp for fp in vol_files}
mask_dict = {get_vol_idx(fp): fp for fp in mask_files}
common_idx = sorted(set(vol_dict) & set(mask_dict))
vol_mask_pairs = [(vol_dict[i], mask_dict[i]) for i in common_idx]

assert len(vol_mask_pairs) > 0, '볼륨-라벨 쌍을 찾을 수 없습니다. 경로를 확인하세요.'
print(f'매칭된 CT 볼륨 수: {len(vol_mask_pairs)}')

# 1-3. 슬라이스 전처리 및 .npz 저장 (최초 1회, 이후 자동 스킵)
SLICE_DIR = '/tmp/lits_slices'
os.makedirs(SLICE_DIR, exist_ok=True)

def hu_window_normalize(arr):
    """HU windowing [-100, 250] → [0, 1] 정규화"""
    arr = np.clip(arr, HU_MIN, HU_MAX)
    arr = (arr - HU_MIN) / (HU_MAX - HU_MIN)
    return arr.astype(np.float32)

existing_slices = glob.glob(os.path.join(SLICE_DIR, '*.npz'))
if len(existing_slices) < 500:
    print('슬라이스 전처리 중 (최초 1회 실행, 이후 캐시 사용)...')
    for vol_path, mask_path in tqdm(vol_mask_pairs, desc='Processing volumes'):
        idx = get_vol_idx(vol_path)
        vol_arr  = nib.load(vol_path).get_fdata().astype(np.float32)   # (H, W, D)
        mask_arr = nib.load(mask_path).get_fdata().astype(np.int64)    # (H, W, D)
        vol_arr  = hu_window_normalize(vol_arr)

        n_slices = vol_arr.shape[2]
        for s in range(n_slices):
            if mask_arr[:, :, s].max() == 0:   # liver 없는 슬라이스 제외
                continue
            img_s  = cv2.resize(vol_arr[:, :, s],  (IMG_SIZE, IMG_SIZE),
                                interpolation=cv2.INTER_LINEAR)
            mask_s = cv2.resize(mask_arr[:, :, s], (IMG_SIZE, IMG_SIZE),
                                interpolation=cv2.INTER_NEAREST)
            np.savez_compressed(
                os.path.join(SLICE_DIR, f'vol{idx:03d}_s{s:04d}.npz'),
                image=img_s.astype(np.float32),
                label=mask_s.astype(np.int64)
            )
    print('전처리 완료')
else:
    print(f'캐시 사용: {len(existing_slices)}개 슬라이스 이미 존재')

slice_files = sorted(glob.glob(os.path.join(SLICE_DIR, '*.npz')))
print(f'총 슬라이스 수 (liver 포함): {len(slice_files)}')

# 1-4. Train / Val / Test 분할 — 볼륨 단위 (data leakage 방지, 8:1:1)
vol_ids = sorted(set(int(os.path.basename(f).split('_')[0][3:]) for f in slice_files))
tr_vol_ids, tmp_vol_ids = train_test_split(vol_ids, test_size=0.2, random_state=SEED)
val_vol_ids, test_vol_ids = train_test_split(tmp_vol_ids, test_size=0.5, random_state=SEED)
tr_vol_set   = set(tr_vol_ids)
val_vol_set  = set(val_vol_ids)
test_vol_set = set(test_vol_ids)

tr_files   = [f for f in slice_files if int(os.path.basename(f).split('_')[0][3:]) in tr_vol_set]
val_files  = [f for f in slice_files if int(os.path.basename(f).split('_')[0][3:]) in val_vol_set]
test_files = [f for f in slice_files if int(os.path.basename(f).split('_')[0][3:]) in test_vol_set]

print(f'Train: {len(tr_files)} slices ({len(tr_vol_ids)} vols)')
print(f'Val  : {len(val_files)} slices ({len(val_vol_ids)} vols)')
print(f'Test : {len(test_files)} slices ({len(test_vol_ids)} vols)')

# 1-5. Dataset 클래스
class LiTSDataset(Dataset):
    """
    LiTS 2D Axial Slice Dataset.
    Label: 0=Background, 1=Liver, 2=Tumor
    Input: 3-channel (grayscale stacked) for ImageNet-pretrained encoder
    """
    def __init__(self, npz_files, augment=False):
        self.files   = npz_files
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data  = np.load(self.files[idx])
        image = data['image'].astype(np.float32)   # (H, W), [0, 1]
        label = data['label'].astype(np.int64)     # (H, W), {0, 1, 2}

        if self.augment:
            if random.random() > 0.5:
                image = np.fliplr(image).copy()
                label = np.fliplr(label).copy()
            if random.random() > 0.5:
                image = np.flipud(image).copy()
                label = np.flipud(label).copy()
            k = random.randint(0, 3)
            image = np.rot90(image, k).copy()
            label = np.rot90(label, k).copy()

        # ImageNet 정규화 (grayscale → 3ch 복제)
        image = (image - 0.456) / 0.224
        image = np.stack([image, image, image], axis=0).astype(np.float32)  # (3, H, W)

        return torch.from_numpy(image), torch.from_numpy(label).long()

# 1-6. DataLoader
train_loader = DataLoader(
    LiTSDataset(tr_files,  augment=True),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    LiTSDataset(val_files, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    LiTSDataset(test_files, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
print('DataLoader 구성 완료')

LiTS 데이터셋 다운로드 중...


100%|██████████| 4.84G/4.84G [02:01<00:00, 42.8MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/andrewmvd/liver-tumor-segmentation/versions/5
매칭된 CT 볼륨 수: 51
슬라이스 전처리 중 (최초 1회 실행, 이후 캐시 사용)...


Processing volumes: 100%|██████████| 51/51 [02:39<00:00,  3.12s/it]

전처리 완료
총 슬라이스 수 (liver 포함): 6802
Train: 5433 slices (40 vols)
Val  : 1369 slices (11 vols)
DataLoader 구성 완료


In [3]:
# ── Cell 2: 클래스 비율 계산 (rules.md §5-3: 픽셀 단위, 학습 데이터만) ────────
print('클래스 비율 계산 중 (학습 슬라이스)...')

class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for fp in tqdm(tr_files, desc='Counting pixels'):
    label = np.load(fp)['label'].astype(np.int64)
    for c in range(NUM_CLASSES):
        class_counts[c] += int((label == c).sum())

class_counts = class_counts.tolist()
total = sum(class_counts)

print()
for c, (name, cnt) in enumerate(zip(CLASS_NAMES, class_counts)):
    print(f'  [{c}] {name:<12}: {cnt:>15,} pixels  ({100 * cnt / total:.2f}%)')

print(f'\nBG : Liver  = {class_counts[0] / class_counts[1]:.1f} : 1')
print(f'BG : Tumor  = {class_counts[0] / class_counts[2]:.1f} : 1')
print(f'Liver : Tumor = {class_counts[1] / class_counts[2]:.1f} : 1')
print(f'\nclass_counts = {class_counts}')

클래스 비율 계산 중 (학습 슬라이스)...


Counting pixels: 100%|██████████| 5433/5433 [00:10<00:00, 506.66it/s]


  [0] Background  :     333,333,760 pixels  (93.62%)
  [1] Liver       :      21,787,222 pixels  (6.12%)
  [2] Tumor       :         936,106 pixels  (0.26%)

BG : Liver  = 15.3 : 1
BG : Tumor  = 356.1 : 1
Liver : Tumor = 23.3 : 1

class_counts = [333333760, 21787222, 936106]


In [4]:
# ── Cell 3: 모델 + 유틸리티 함수 ──────────────────────────────────────────────
#
# [SoTA 참고]
#   nnU-Net (3D, Isensee et al.): Liver ~0.963, Tumor ~0.702~0.739
#   U-Net++ (2D, ResNet50):       Liver ~0.956, Tumor ~0.737  ← 본 실험 모델
#   출처: LiTS Benchmark paper (Bilic et al., 2023, Medical Image Analysis)
#
# [선택 이유]
#   - 손실 함수 효과만 분리 측정 → SoTA 2D 모델 고정
#   - smp.UnetPlusPlus(ResNet50, ImageNet) = 재현 가능한 공개 구현
#   - 모델 가중치·하이퍼파라미터는 원 논문 설정 유지, 손실 함수만 교체

def build_model():
    """U-Net++ (ResNet50, ImageNet pretrained) — 3-class CT segmentation."""
    return smp.UnetPlusPlus(
        encoder_name    = 'resnet50',
        encoder_weights = 'imagenet',
        in_channels     = 3,
        classes         = NUM_CLASSES,
        activation      = None,
    ).to(device)


def compute_val_mdice(model, loader):
    """빠른 Val mDice (Liver+Tumor 평균, BG 제외) — 학습 루프 및 Optuna용."""
    model.eval()
    dice_per_class = np.zeros(NUM_CLASSES - 1)
    n_batches = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = torch.argmax(model(imgs), dim=1)
            for c_idx, c in enumerate(range(1, NUM_CLASSES)):
                p = (preds == c).float()
                t = (masks == c).float()
                inter = (p * t).sum()
                union = p.sum() + t.sum()
                if union > 0:
                    dice_per_class[c_idx] += (2. * inter / (union + 1e-8)).item()
            n_batches += 1
    dice_per_class /= max(n_batches, 1)
    return float(np.mean(dice_per_class))


def compute_val_metrics(model, loader):
    """전체 Val 지표: Liver Dice, Tumor Dice, mDice."""
    model.eval()
    dice_per_class = np.zeros(NUM_CLASSES - 1)
    n_batches = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = torch.argmax(model(imgs), dim=1)
            for c_idx, c in enumerate(range(1, NUM_CLASSES)):
                p = (preds == c).float()
                t = (masks == c).float()
                inter = (p * t).sum()
                union = p.sum() + t.sum()
                if union > 0:
                    dice_per_class[c_idx] += (2. * inter / (union + 1e-8)).item()
            n_batches += 1
    dice_per_class /= max(n_batches, 1)
    return {
        'Liver_Dice': float(dice_per_class[0]),
        'Tumor_Dice': float(dice_per_class[1]),
        'mDice':      float(np.mean(dice_per_class)),
    }


# 파라미터 수 확인
test_model = build_model()
n_params   = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f'U-Net++ (ResNet50) 파라미터 수: {n_params:,}')
del test_model
print('모델 + 유틸리티 함수 준비 완료')

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

U-Net++ (ResNet50) 파라미터 수: 48,986,035
모델 + 유틸리티 함수 준비 완료


In [5]:
# ── Cell 4: 학습 함수 ─────────────────────────────────────────────────────────

def train_model(
    loss_name,
    alpha=1.0,
    gamma=2.0,
    epochs=50,
    lr=1e-4,
    subset_ratio=1.0,
    tag='',
):
    """
    U-Net++ 학습 함수.

    Args:
        loss_name:    손실 함수 이름 (예: 'lwce_dice', 'plwce_dice')
        alpha:        PLWCE / PWCE 강도 파라미터
        epochs:       학습 에폭 수
        lr:           학습률
        subset_ratio: Optuna proxy용 데이터 축소 비율 (1.0 = 전체)
        tag:          저장 파일 구분 태그

    Returns:
        Tuple[nn.Module, dict, float]: (최고 모델, 히스토리, 최고 Val mDice)
    """
    model     = build_model()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # 손실 함수 교체 핵심 — 모델 구조·하이퍼파라미터 유지, 손실함수만 변경
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)

    name = f'{loss_name}_alpha{alpha:.2f}' if alpha != 1.0 else loss_name
    if tag:
        name = f'{tag}_{name}'

    print(f"\n{'='*60}\nU-Net++ + {name}  (epochs={epochs})\n{'='*60}")

    # Optuna proxy용 데이터 축소
    if subset_ratio < 1.0:
        n = max(1, int(len(train_loader.dataset) * subset_ratio))
        sub_ds = torch.utils.data.Subset(
            train_loader.dataset,
            random.sample(range(len(train_loader.dataset)), n)
        )
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE,
                            shuffle=True, num_workers=NUM_WORKERS)
    else:
        loader = train_loader

    history    = {'loss': [], 'val_mdice': []}
    best_mdice = 0.0
    save_path  = f'/tmp/best_unetpp_{name}.pth'

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for imgs, masks in tqdm(loader, desc=f'Ep{epoch+1:02d}/{epochs}', leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss  = epoch_loss / len(loader)
        val_mdice = compute_val_mdice(model, val_loader)

        history['loss'].append(avg_loss)
        history['val_mdice'].append(val_mdice)

        print(f'Ep{epoch+1:02d} | Loss: {avg_loss:.4f} | Val mDice: {val_mdice:.4f}', end='')
        if val_mdice > best_mdice:
            best_mdice = val_mdice
            torch.save(model.state_dict(), save_path)
            print('  <- Best!', end='')
        print()

    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'최고 Val mDice: {best_mdice:.4f}')
    return model, history, best_mdice


print('train_model() 함수 준비 완료')

train_model() 함수 준비 완료


In [6]:
# ── Cell 5: Optuna alpha 탐색 (rules.md §5-1, §6-4) ──────────────────────────
#   plwce: alpha 범위 2.5 ~ 15.0  (log 스케일이 이미 작아 큰 alpha 필요)
#   pwce:  alpha 범위 0.2 ~ 2.5   ((N/n)^α 기저값이 크므로 작은 alpha로 제한)

ALPHA_LOW_PLWCE,  ALPHA_HIGH_PLWCE  = 2.5,  15.0
ALPHA_LOW_PWCE,   ALPHA_HIGH_PWCE   = 0.2,  2.5
PROXY_EPOCHS = 5
PROXY_SUBSET = 0.15
N_TRIALS     = 20
N_TRIALS_PF  = N_TRIALS * 2  # PLWCE+Focal: 파라미터 2개(alpha,gamma)이므로 2배


def make_objective(loss_name, alpha_low, alpha_high):
    """loss_name 별 Optuna objective 생성."""
    def objective(trial):
        alpha = trial.suggest_float('alpha', alpha_low, alpha_high)
        try:
            _, _, mdice = train_model(
                loss_name    = loss_name,
                alpha        = alpha,
                epochs       = PROXY_EPOCHS,
                subset_ratio = PROXY_SUBSET,
                tag          = f'trial{trial.number}',
            )
            return mdice
        except Exception as e:
            print(f'Trial {trial.number} 실패: {e}')
            return 0.0
    return objective


# ── PLWCE alpha 탐색 ──────────────────────────────────────────────────────────
print(f'[Optuna] PLWCE alpha 탐색  (범위: {ALPHA_LOW_PLWCE}~{ALPHA_HIGH_PLWCE}, {N_TRIALS} trials)')
study_plwce = optuna.create_study(
    direction   = 'maximize',
    study_name  = 'unetpp_lits_plwce_alpha',
    pruner      = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study_plwce.optimize(make_objective('plwce_dice', ALPHA_LOW_PLWCE, ALPHA_HIGH_PLWCE), n_trials=N_TRIALS)
best_alpha_plwce = study_plwce.best_params['alpha']
print(f'[PLWCE] 최적 alpha = {best_alpha_plwce:.4f}  (Val mDice = {study_plwce.best_value:.4f})')

# ── PWCE alpha 탐색 ───────────────────────────────────────────────────────────
print(f'\n[Optuna] PWCE alpha 탐색  (범위: {ALPHA_LOW_PWCE}~{ALPHA_HIGH_PWCE}, {N_TRIALS} trials)')
study_pwce = optuna.create_study(
    direction   = 'maximize',
    study_name  = 'unetpp_lits_pwce_alpha',
    pruner      = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study_pwce.optimize(make_objective('pwce_dice', ALPHA_LOW_PWCE, ALPHA_HIGH_PWCE), n_trials=N_TRIALS)
best_alpha_pwce = study_pwce.best_params['alpha']
print(f'[PWCE]  최적 alpha = {best_alpha_pwce:.4f}  (Val mDice = {study_pwce.best_value:.4f})')

# ── PLWCE+Focal alpha + gamma 공동 탐색 ─────────────────────────────────────
ALPHA_LOW_PF, ALPHA_HIGH_PF = 2.0, 15.0
GAMMA_LOW_PF, GAMMA_HIGH_PF = 0.0,  5.0

print(f'\n[Optuna] PLWCE+Focal alpha+gamma 탐색  '
      f'(alpha: {ALPHA_LOW_PF}~{ALPHA_HIGH_PF}, gamma: {GAMMA_LOW_PF}~{GAMMA_HIGH_PF}, {N_TRIALS} trials)')

def objective_pf(trial):
    alpha = trial.suggest_float('alpha', ALPHA_LOW_PF, ALPHA_HIGH_PF)
    gamma = trial.suggest_float('gamma', GAMMA_LOW_PF, GAMMA_HIGH_PF)
    try:
        _, _, dice = train_model(
            loss_name    = 'plwce_focal_dice',
            alpha        = alpha,
            gamma        = gamma,
            epochs       = PROXY_EPOCHS,
            subset_ratio = PROXY_SUBSET,
            tag          = f'trial{trial.number}',
        )
        return dice
    except Exception as e:
        print(f'Trial {trial.number} 실패: {e}')
        return 0.0

study_pf = optuna.create_study(
    direction  = 'maximize',
    study_name = f'unet_{DOMAIN}_plwce_focal_alpha_gamma',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study_pf.optimize(objective_pf, n_trials=N_TRIALS_PF)
best_alpha_pf = study_pf.best_params['alpha']
best_gamma_pf = study_pf.best_params['gamma']
print(f'[PLWCE+Focal] 최적 alpha={best_alpha_pf:.4f}, gamma={best_gamma_pf:.4f}  '
      f'(Val Dice = {study_pf.best_value:.4f})')

# ── Optuna 결과 저장 ──────────────────────────────────────────────────────────
optuna_results = {
    'plwce': {
        'best_alpha': best_alpha_plwce,
        'best_proxy_mdice': study_plwce.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'), 'value': t.value}
                   for t in study_plwce.trials if t.value is not None],
    },
    'pwce': {
        'best_alpha': best_alpha_pwce,
        'best_proxy_mdice': study_pwce.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'), 'value': t.value}
                   for t in study_pwce.trials if t.value is not None],
    },
    'plwce_focal': {
        'best_alpha': best_alpha_pf,
        'best_gamma': best_gamma_pf,
        'best_proxy_dice': study_pf.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'),
                    'gamma': t.params.get('gamma'), 'value': t.value}
                   for t in study_pf.trials if t.value is not None],
    },
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_results.json'), 'w') as f:
    json.dump(optuna_results, f, indent=2, ensure_ascii=False)
print(f'\nOptuna 결과 저장: {os.path.join(RESULTS_DIR, f"{DOMAIN}_optuna_results.json")}')

# ── 탐색 결과 시각화 ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, study, sname, a_range in [
    (axes[0], study_plwce, 'PLWCE', f'{ALPHA_LOW_PLWCE}~{ALPHA_HIGH_PLWCE}'),
    (axes[1], study_pwce,  'PWCE',  f'{ALPHA_LOW_PWCE}~{ALPHA_HIGH_PWCE}'),
]:
    trials = [t for t in study.trials if t.value is not None]
    alphas = [t.params['alpha'] for t in trials]
    values = [t.value for t in trials]
    best_a = study.best_params['alpha']
    best_v = study.best_value

    ax.scatter(alphas, values, alpha=0.5, s=40, label='Trials')
    ax.axvline(best_a, color='red', linestyle='--', label=f'Best alpha={best_a:.2f}')
    ax.scatter([best_a], [best_v], color='red', s=100, zorder=5)
    ax.set_xlabel('alpha'); ax.set_ylabel('Val mDice (proxy)')
    ax.set_title(f'{sname} alpha 탐색 (범위 {a_range})')
    ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_search.png'), dpi=100)
plt.show()

# PLWCE+Focal: alpha vs gamma 2D 탐색 결과 시각화
fig_pf, ax_pf = plt.subplots(1, 1, figsize=(7, 5))
pf_trials = [t for t in study_pf.trials if t.value is not None]
pf_alphas = [t.params['alpha'] for t in pf_trials]
pf_gammas = [t.params['gamma'] for t in pf_trials]
pf_values = [t.value for t in pf_trials]
sc = ax_pf.scatter(pf_alphas, pf_gammas, c=pf_values, cmap='viridis', alpha=0.7, s=60)
ax_pf.scatter([best_alpha_pf], [best_gamma_pf], color='red', s=150, zorder=5,
              marker='*', label=f'Best α={best_alpha_pf:.2f}, γ={best_gamma_pf:.2f}')
plt.colorbar(sc, ax=ax_pf, label='Val Dice')
ax_pf.set_xlabel('alpha'); ax_pf.set_ylabel('gamma')
ax_pf.set_title('PLWCE+Focal alpha+gamma 탐색')
ax_pf.legend(fontsize=8); ax_pf.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_search_pf.png'), dpi=100)
plt.show()
print(f'PLWCE+Focal 탐색 결과 저장: {RESULTS_DIR}/{DOMAIN}_optuna_search_pf.png')print(f'탐색 결과 저장: {os.path.join(RESULTS_DIR, f"{DOMAIN}_optuna_search.png")}')

[Optuna] PLWCE alpha 탐색  (범위: 2.5~15.0, 20 trials)


[plwce_dice] Weights (plwce): Generated.

U-Net++ + trial0_plwce_dice_alpha13.49  (epochs=5)


Ep01 | Loss: 0.8761 | Val mDice: 0.1575  <- Best!


Ep02 | Loss: 0.6635 | Val mDice: 0.3247  <- Best!


Ep03 | Loss: 0.5835 | Val mDice: 0.3884  <- Best!


...

Ep01 | Loss: 0.7170 | Val mDice: 0.3919  <- Best!


Ep02 | Loss: 0.5316 | Val mDice: 0.4380  <- Best!


Ep03 | Loss: 0.4586 | Val mDice: 0.4716  <- Best!


Ep04 | Loss: 0.4260 | Val mDice: 0.4825  <- Best!


Ep05 | Loss: 0.4090 | Val mDice: 0.4835  <- Best!
최고 Val mDice: 0.4835
[PWCE]  최적 alpha = 0.5406  (Val mDice = 0.4935)

Optuna 결과 저장: /root/imbalanced-data-LWCE/medical_data/results/lits_optuna_results.json
탐색 결과 저장: /root/imbalanced-data-LWCE/medical_data/results/lits_optuna_search.png


In [7]:
# ── Cell 6: 전체 Loss 비교 실험 ───────────────────────────────────────────────
# rules.md §5-1: ce_dice 기준선 + 최소 4종 이상 비교

FINAL_EPOCHS = 50
FINAL_LR     = 1e-4

# Optuna 결과 로드 (Cell 5 미실행 시 JSON fallback)
try:
    _ = best_alpha_plwce
except NameError:
    try:
        with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_results.json')) as f:
            d = json.load(f)
        best_alpha_plwce = d['plwce']['best_alpha']
        best_alpha_pwce  = d['pwce']['best_alpha']
        best_alpha_pf    = d.get('plwce_focal', {}).get('best_alpha', 7.0)
        best_gamma_pf    = d.get('plwce_focal', {}).get('best_gamma', 2.0)
        print(f'Optuna 결과 로드: PLWCE alpha={best_alpha_plwce:.4f}, PWCE alpha={best_alpha_pwce:.4f}')
    except FileNotFoundError:
        best_alpha_plwce = 7.0
        best_alpha_pwce  = 0.5
        best_alpha_pf    = 7.0
        best_gamma_pf    = 2.0
        print('Optuna 미실행 → 기본값 사용 (PLWCE alpha=7.0, PWCE alpha=0.5)')

# ── 실험 목록 ─────────────────────────────────────────────────────────────────
experiments = [
    ('ce_dice',          1.0,              2.0,             'CE+Dice              (기준선)'),
    ('wce_dice',         1.0,              2.0,             'WCE+Dice'),
    ('lwce_dice',        1.0,              2.0,             'LWCE+Dice'),
    ('plwce_dice',       best_alpha_plwce, 2.0,             f'PLWCE+Dice           (alpha={best_alpha_plwce:.2f})'),
    ('cb_dice',          1.0,              2.0,             'CB+Dice'),
    ('plwce_focal_dice', best_alpha_pf,    best_gamma_pf,   f'PLWCE+Focal+Dice     (α={best_alpha_pf:.2f}, γ={best_gamma_pf:.2f})'),
]

all_results = {}
for loss_name, alpha, gamma, label in experiments:
    model, history, best_mdice = train_model(
        loss_name = loss_name,
        alpha     = alpha,
        gamma     = gamma,
        epochs    = FINAL_EPOCHS,
        lr        = FINAL_LR,
        tag       = 'final',
    )
    all_results[label] = {
        'model':      model,
        'history':    history,
        'best_mdice': best_mdice,
        'loss_name':  loss_name,
        'alpha':      alpha,
        'gamma':      gamma,
    }

# ── 1차 요약 ─────────────────────────────────────────────────────────────────
print('\n' + '='*50)
print('[Loss 비교 실험 1차 요약 — Val mDice]')
print(f"{'Loss':<35} {'Best Val mDice':>14}")
print('-' * 51)
for label, v in all_results.items():
    print(f"{label:<35} {v['best_mdice']:>14.4f}")


U-Net++ + final_ce_dice  (epochs=50)


Ep01 | Loss: 0.5771 | Val mDice: 0.4811  <- Best!


Ep02 | Loss: 0.2382 | Val mDice: 0.5021  <- Best!


Ep03 | Loss: 0.1177 | Val mDice: 0.5155  <- Best!


Ep04 | Loss: 0.0933 | Val mDice: 0.4878


...

Ep46 | Loss: 0.0347 | Val mDice: 0.5208


Ep47 | Loss: 0.0332 | Val mDice: 0.5201


Ep48 | Loss: 0.0377 | Val mDice: 0.5219


Ep49 | Loss: 0.0356 | Val mDice: 0.5215


Ep50 | Loss: 0.0337 | Val mDice: 0.5215
최고 Val mDice: 0.5287

[Loss 비교 실험 1차 요약 — Val mDice]
Loss                                Best Val mDice
---------------------------------------------------
CE+Dice        (기준선)                        0.5342
WCE+Dice                                    0.5348
LWCE+Dice                                   0.5404
PLWCE+Dice     (alpha=8.49)                 0.5310
CB+Dice                                     0.5287


In [8]:
# ── Cell 7: 시각화 — 학습 곡선 + 예측 결과 ───────────────────────────────────

COLORS = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F', '#B47CC7']

# 7-1. 학습 곡선 (Train Loss + Val mDice)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for i, (label, v) in enumerate(all_results.items()):
    h = v['history']
    ax1.plot(h['loss'],     label=label, color=COLORS[i % len(COLORS)])
    ax2.plot(h['val_mdice'], label=label, color=COLORS[i % len(COLORS)])

ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch')
ax1.legend(fontsize=7); ax1.grid(True)

ax2.set_title('Val mDice (Liver + Tumor)'); ax2.set_xlabel('Epoch')
ax2.legend(fontsize=7); ax2.grid(True)

plt.suptitle('LiTS — U-Net++ 학습 곡선 비교', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_training_curves.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'학습 곡선 저장: {os.path.join(RESULTS_DIR, f"{DOMAIN}_training_curves.png")}')

# 7-2. 예측 결과 시각화 (최고 Val mDice 모델, 4열: Input/GT/Tumor Prob/Pred)
best_label = max(all_results, key=lambda k: all_results[k]['best_mdice'])
best_model = all_results[best_label]['model']
best_model.eval()
print(f'\n시각화 모델: {best_label}  (Val mDice={all_results[best_label]["best_mdice"]:.4f})')

# 컬러맵: BG=검정, Liver=노랑, Tumor=빨강
LABEL_COLORS = np.array([[0, 0, 0], [220, 180, 0], [220, 50, 50]], dtype=np.uint8)

def mask_to_rgb(mask_arr):
    """정수 마스크 → RGB 컬러 이미지"""
    rgb = LABEL_COLORS[mask_arr.clip(0, NUM_CLASSES - 1)]
    return rgb

val_ds      = LiTSDataset(val_files, augment=False)
vis_indices = random.sample(range(len(val_ds)), min(4, len(val_ds)))

fig, axes = plt.subplots(len(vis_indices), 4, figsize=(18, len(vis_indices) * 4))

for row, idx in enumerate(vis_indices):
    img_tensor, mask_tensor = val_ds[idx]
    img_np  = img_tensor[0].numpy()    # grayscale (1st channel)
    mask_np = mask_tensor.numpy()

    with torch.no_grad():
        logit = best_model(img_tensor.unsqueeze(0).to(device))  # (1, 3, H, W)
        pred  = torch.argmax(logit, dim=1).squeeze().cpu().numpy()
        prob_tumor = torch.softmax(logit, dim=1)[0, 2].cpu().numpy()  # Tumor 확률맵

    axes[row, 0].imshow(img_np, cmap='gray')
    axes[row, 0].set_title('Input CT Slice'); axes[row, 0].axis('off')

    axes[row, 1].imshow(mask_to_rgb(mask_np))
    axes[row, 1].set_title('Ground Truth (Yellow=Liver, Red=Tumor)'); axes[row, 1].axis('off')

    axes[row, 2].imshow(prob_tumor, cmap='jet', vmin=0, vmax=1)
    axes[row, 2].set_title('Tumor Probability Map'); axes[row, 2].axis('off')

    axes[row, 3].imshow(mask_to_rgb(pred))
    axes[row, 3].set_title(f'Prediction ({best_label.split("(")[0].strip()[:12]})')
    axes[row, 3].axis('off')

plt.suptitle(f'LiTS — 예측 결과 시각화 ({best_label})', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_prediction_vis.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'예측 결과 저장: {os.path.join(RESULTS_DIR, f"{DOMAIN}_prediction_vis.png")}')

학습 곡선 저장: /root/imbalanced-data-LWCE/medical_data/results/lits_training_curves.png

시각화 모델: LWCE+Dice  (Val mDice=0.5404)
예측 결과 저장: /root/imbalanced-data-LWCE/medical_data/results/lits_prediction_vis.png


In [9]:
# ── Cell 8: 최종 정량 평가 + JSON + Excel 저장 (rules.md §5-4) ───────────────

print('\n[전체 모델 종합 평가 — Test Set]')
print(f"{'Loss':<35} {'Liver Dice':>10} {'Tumor Dice':>10} {'mDice':>8}")
print('-' * 65)

final_results = {}
for label, v in all_results.items():
    metrics = compute_val_metrics(v['model'], test_loader)
    final_results[label] = {
        'loss_name':      v['loss_name'],
        'alpha':          v['alpha'],
        'best_val_mdice': v['best_mdice'],
        **metrics,
    }
    print(
        f"{label:<35} "
        f"{metrics['Liver_Dice']:>10.4f} "
        f"{metrics['Tumor_Dice']:>10.4f} "
        f"{metrics['mDice']:>8.4f}"
    )

# ── 바차트 비교 ───────────────────────────────────────────────────────────────
metric_keys  = ['Liver_Dice', 'Tumor_Dice', 'mDice']
metric_names = ['Liver Dice', 'Tumor Dice', 'mDice']
labels_      = list(final_results.keys())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, mkey, mname in zip(axes, metric_keys, metric_names):
    scores = [final_results[lb][mkey] for lb in labels_]
    bars   = ax.bar(range(len(labels_)), scores, color=COLORS[:len(labels_)], alpha=0.85)
    ax.set_xticks(range(len(labels_)))
    ax.set_xticklabels(
        [lb.split('(')[0].strip()[:12] for lb in labels_],
        rotation=30, ha='right', fontsize=8
    )
    ax.set_title(mname); ax.set_ylim(0, 1.05); ax.grid(axis='y', alpha=0.4)
    best_idx = int(np.argmax(scores))
    bars[best_idx].set_edgecolor('red'); bars[best_idx].set_linewidth(2.5)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{score:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('LiTS — Loss별 최종 평가 지표 비교 (Test Set)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_final_metrics.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'평가 차트 저장: {os.path.join(RESULTS_DIR, f"{DOMAIN}_final_metrics.png")}')

# ── JSON 저장 ─────────────────────────────────────────────────────────────────
save_data = {
    'domain':      'LiTS 2017 Liver Tumor Segmentation',
    'model':       'U-Net++ (ResNet50, ImageNet pretrained)',
    'sota_ref': {
        'nnUNet_3D': {'Liver_Dice': 0.963, 'Tumor_Dice': '0.702~0.739'},
        'UNetPP_2D': {'Liver_Dice': '~0.956', 'Tumor_Dice': '~0.737'},
    },
    'num_classes':  NUM_CLASSES,
    'class_counts': {n: int(c) for n, c in zip(CLASS_NAMES, class_counts)},
    'imbalance': {
        'BG_Liver':    round(class_counts[0] / class_counts[1], 1),
        'BG_Tumor':    round(class_counts[0] / class_counts[2], 1),
        'Liver_Tumor': round(class_counts[1] / class_counts[2], 1),
    },
    'hu_window':    [HU_MIN, HU_MAX],
    'train_slices': len(tr_files),
    'val_slices':   len(val_files),
    'final_epochs': FINAL_EPOCHS,
    'results': {
        k: {mk: float(mv) if isinstance(mv, (float, np.floating)) else mv
            for mk, mv in v.items() if mk != 'model'}
        for k, v in final_results.items()
    },
    'best_model': max(final_results, key=lambda k: final_results[k]['mDice']),
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_final_results.json'), 'w', encoding='utf-8') as f:
    json.dump(save_data, f, indent=2, ensure_ascii=False)
print(f'JSON 저장: {os.path.join(RESULTS_DIR, f"{DOMAIN}_final_results.json")}')

# ── Excel 저장 (rules.md §5-4: Summary + Training_History 시트) ──────────────
# Sheet 1: Summary — 손실함수별 최종 파라미터 + 평가지표 한눈에 비교
summary_rows = []
for label, v in final_results.items():
    summary_rows.append({
        'Loss_Function':  label,
        'loss_name':      v['loss_name'],
        'alpha':          round(float(v['alpha']), 4),
        'Best_Val_mDice': round(v['best_val_mdice'], 4),
        'Test_Liver_Dice': round(v['Liver_Dice'], 4),
        'Test_Tumor_Dice': round(v['Tumor_Dice'], 4),
        'Test_mDice':      round(v['mDice'], 4),
        'BG_Liver_ratio': round(class_counts[0] / class_counts[1], 1),
        'BG_Tumor_ratio': round(class_counts[0] / class_counts[2], 1),
        'HU_min':         HU_MIN,
        'HU_max':         HU_MAX,
        'epochs':         FINAL_EPOCHS,
        'model':          'U-Net++ (ResNet50)',
    })
df_summary = pd.DataFrame(summary_rows)

# Sheet 2: Training_History — 에폭별 Loss·Val mDice 추이
history_rows = []
for label, v in all_results.items():
    for ep, (loss, mdice) in enumerate(
        zip(v['history']['loss'], v['history']['val_mdice']), 1
    ):
        history_rows.append({
            'Loss_Function': label,
            'Epoch':         ep,
            'Train_Loss':    round(loss,  6),
            'Test_mDice':     round(mdice, 6),
        })
df_history = pd.DataFrame(history_rows)

excel_path = os.path.join(RESULTS_DIR, f'{DOMAIN}_final_results.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary', index=False)
    df_history.to_excel(writer, sheet_name='Training_History', index=False)
print(f'Excel 저장: {excel_path}')

# ── 최종 요약 출력 ────────────────────────────────────────────────────────────
print(f"\n최고 모델: {save_data['best_model']}")
print(f'\n[불균형 비율]')
print(f'  BG : Liver  = {save_data["imbalance"]["BG_Liver"]:>6.1f} : 1')
print(f'  BG : Tumor  = {save_data["imbalance"]["BG_Tumor"]:>6.1f} : 1')
print(f'  Liver : Tumor = {save_data["imbalance"]["Liver_Tumor"]:>4.1f} : 1')


[전체 모델 종합 평가 — Val Set]
Loss                                Liver Dice Tumor Dice    mDice
-----------------------------------------------------------------
CE+Dice        (기준선)                    0.9410     0.1274   0.5342
WCE+Dice                                0.9246     0.1449   0.5348
LWCE+Dice                               0.9271     0.1537   0.5404
PLWCE+Dice     (alpha=8.49)             0.9266     0.1354   0.5310
CB+Dice                                 0.9254     0.1319   0.5287
평가 차트 저장: /root/imbalanced-data-LWCE/medical_data/results/lits_final_metrics.png
JSON 저장: /root/imbalanced-data-LWCE/medical_data/results/lits_final_results.json
Excel 저장: /root/imbalanced-data-LWCE/medical_data/results/lits_final_results.xlsx

최고 모델: LWCE+Dice

[불균형 비율]
  BG : Liver  =   15.3 : 1
  BG : Tumor  =  356.1 : 1
  Liver : Tumor = 23.3 : 1
